In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
import logging
import baccoemu
from emantis.matter_power_spectrum import NonLinearMGBoostEmulator
from scipy.interpolate import interp1d
from scipy.integrate import simpson, quad
import os
from contextlib import redirect_stdout, redirect_stderr

warnings.simplefilter("ignore")
logging.getLogger("py.warnings").setLevel(logging.CRITICAL)

class Config:
    COSMO_PARAMS = {
        "Omega_m": 0.315,   # Total matter density parameter
        "Omega_b": 0.05,    # Baryon density parameter
        "h": 0.67,          # Reduced Hubble constant (H0 = 100h km/s/Mpc)
        "n_s": 0.96,        # Primordial scalar spectral index
        "sigma8": 0.83,     # RMS matter fluctuation amplitude at 8 Mpc/h
        "M_nu": 0.0,        # Sum of neutrino masses (eV)
        "w0": -1.0,         # Dark energy equation of state today
        "wa": 0.0,          # Time variation of dark energy equation of state
        "aexp": 1.0         # Scale factor (1.0 = present day)
    }

    TRANSFER_FUNCTION = "boltzmann_camb"  # Use Boltzmann solver (e.g., CAMB) for transfer function

    c = 299792.458        # Speed of light in km/s

# define the redshift grid for integrals

    Z_MIN = 0.01          # Minimum redshift
    Z_MAX = 3.0           # Maximum redshift
    N_Z = 500             # Number of redshift sampling points
    Z0_GALAXY = 0.3       # Characteristic galaxy redshift parameter

    ELL_MIN = 10          # Minimum multipole ℓ
    ELL_MAX = 2000        # Maximum multipole ℓ
    N_ELL = 50            # Number of ℓ sampling points

    BIAS_B0 = 2.0         # Linear galaxy bias parameter

    alpha = 2.225         # Magnification bias parameter (Karim et al., α ≃ 2.225)

    FR_VALUES = [4, 5, 6] # f(R) gravity parameter values (e.g., log10|fR0|)

    BACCO_K_MIN = -2      # Minimum log10(k) for power spectrum grid
    BACCO_K_MAX = None    # Maximum log10(k) (None = use default)
    BACCO_N_K = 200       # Number of k sampling points
    EMANTIS_VERBOSE = False  # Toggle emulator verbosity/logging

    FIGURE_SIZE = (14, 10)   

    K_MIN_SAFETY = 1.1       # Safety factor for minimum k boundary
    K_MAX_SAFETY = 0.9       # Safety factor for maximum k boundary
    K_CLIP_THRESHOLD = 0.05  # Threshold for clipping unstable k values


# ============================================================
# Core Cosmological Functions
# ============================================================

# Hubble Function (Eq. 1, Lin et al., 2009)
def hubble_function(z, H0, Om_m, Om_lambda):
    return H0 * np.sqrt(
        Om_m * (1 + z)**3 +
        (1 - Om_m - Om_lambda) * (1 + z)**2 +
        Om_lambda
    )

# Comoving Distance (Eq. 146, Lahav et al., 2004)
def comoving_distance_proper(z, H0, Om_m, Om_lambda):
    def integrand(z_prime):
        return Config.c / hubble_function(z_prime, H0, Om_m, Om_lambda)
    
    # Handle both scalar and array inputs
    if np.isscalar(z):
        chi, _ = quad(integrand, 0, z)
        return chi
    else:
        return np.array([quad(integrand, 0, z_val)[0] for z_val in z])


# Omega(z) (Eq. 68, Lahav et al., 2004)
def compute_Omega_z(z, H0, Om_m, Om_lambda):
    num = Om_m * (1 + z)**3
    den = Om_m * (1 + z)**3 + (1 - Om_m - Om_lambda) * (1 + z)**2 + Om_lambda
    return num / den


# Lambda(z) (Eq. 69, Lahav et al., 2004)
def compute_lambda_z(z, H0, Om_m, Om_lambda):
    num = Om_lambda
    den = Om_m * (1 + z)**3 + (1 - Om_m - Om_lambda) * (1 + z)**2 + Om_lambda
    return num / den  


# Growth Function g(z) (Eq. 67, Lahav et al., 2004)
def growth_function_g(z, H0, Om_m, Om_lambda):
    Omega_z = compute_Omega_z(z, H0, Om_m, Om_lambda)
    lambda_z = compute_lambda_z(z, H0, Om_m, Om_lambda)
    den = Omega_z**(4/7) - lambda_z + (1 + Omega_z/2) * (1 + lambda_z/70)
    return (5 * Omega_z / 2) * (1 / den)


def growth_factor_D(z, H0, Om_m, Om_lambda):  #need to modify
    """
    Linear growth factor D(z), normalized so that D(0) = 1.

    Lahav et al. (2004), Eq. (66):
    D_raw(z) = g(z) / (1 + z)

    We then normalize:
    D(z) = D_raw(z) / D_raw(0)
         = [g(z)/(1+z)] / [g(0)/1]
         = g(z) / [(1+z) * g(0)]
    """
    
    # Handle both scalar and array inputs
    if np.isscalar(z):
        # ---- Step 1: Compute raw (unnormalized) growth ----
        g_z = growth_function_g(z, H0, Om_m, Om_lambda)
        D_z_raw = g_z / (1.0 + z)

        # ---- Step 2: Compute today's raw value ----
        g_0 = growth_function_g(0.0, H0, Om_m, Om_lambda)
        D_0_raw = g_0  # since (1+0) = 1

        # ---- Step 3: normalize so D(0) = 1 ----
        return D_z_raw / D_0_raw
    else:
        # For arrays, vectorize the computation
        g_z = np.array([growth_function_g(z_val, H0, Om_m, Om_lambda) for z_val in z])
        D_z_raw = g_z / (1.0 + z)
        
        g_0 = growth_function_g(0.0, H0, Om_m, Om_lambda)
        D_0_raw = g_0
        
        return D_z_raw / D_0_raw


# CMB Lensing Kernel W_k(z) (Eq. 4.2, Karim et al., 2025) (#recheck)
def cmb_lensing_kernel(z, z_star, H0, Om_m, Om_lambda):
    prefactor = (3 * Om_m * H0**2) / (2 * Config.c)
    
    # Handle both scalar and array inputs
    if np.isscalar(z):
        chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
        chi_star = comoving_distance_proper(z_star, H0, Om_m, Om_lambda)
        return prefactor * (1 + z) * chi_z * (chi_star - chi_z) / chi_star
    else:
        chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
        chi_star = comoving_distance_proper(z_star, H0, Om_m, Om_lambda)
        return prefactor * (1 + z) * chi_z * (chi_star - chi_z) / chi_star


# Galaxy Lensing Kernel W_g(z) (Eq. 4.3 & 4.4, Karim et al., 2025) (#recheck)
def galaxy_lensing_kernel(z, z_array, dN_dz_func, bias_func, H0, Om_m, Om_lambda):
    bias_term = bias_func(z) * dN_dz_func(z)
    chi_z = comoving_distance_proper(z, H0, Om_m, Om_lambda)
    def integrand(z_prime):
        chi_z_prime = comoving_distance_proper(z_prime, H0, Om_m, Om_lambda)
        return (1 - chi_z / chi_z_prime) * (Config.alpha - 1) * dN_dz_func(z_prime)
    z_star = 10.0
    integral_term, _ = quad(integrand, z, z_star)
    mu_prefactor = (3 * Om_m * H0**2) / (2 * Config.c) * (1 + z) * chi_z
    mu_z = mu_prefactor * integral_term
    return bias_term + mu_z


# Bias Function (Eq. 4.7, Karim et al., 2025) #D∗(z) represents the normalized growth factor
def bias_function(z, H0, Omega_m, Omega_lambda, b0=1.0):
    D_z = growth_factor_D(z, H0, Omega_m, Omega_lambda)
    D_0 = growth_factor_D(0, H0, Omega_m, Omega_lambda)
    D_star_z = D_z / D_0
    return b0 / (D_star_z + 1e-30)


# ============================================================
# Power Spectrum Utilities (BACCO / EMANTIS)
# ============================================================

def initialize_bacco_emulator(params, k_min=-2, k_max=None, n_k=200, verbose=False):
    """Initialize BACCO emulator for matter power spectrum."""
    with open(os.devnull, 'w') as fnull:
        with redirect_stdout(fnull), redirect_stderr(fnull):
            emulator = baccoemu.Matter_powerspectrum(verbose=verbose)

            if k_max is None:
                k_max = np.log10(emulator.emulator['nonlinear']['k'].max())

            k = np.logspace(k_min, k_max, num=n_k)

            params_bacco = {
                "omega_cold": params["Omega_m"] - params["Omega_b"],
                "sigma8_cold": params["sigma8"],
                "omega_baryon": params["Omega_b"],
                "ns": params["n_s"],
                "hubble": params["h"],
                "neutrino_mass": params.get("M_nu", 0.0),
                "w0": params.get("w0", -1.0),
                "wa": params.get("wa", 0.0),
                "expfactor": params.get("aexp", 1.0)
            }

            k, Q_boost = emulator.get_nonlinear_boost(k=k, cold=False, **params_bacco)
            k, pk_nl = emulator.get_nonlinear_pk(k=k, cold=False, **params_bacco)

    return k, pk_nl, Q_boost


def initialize_emantis_emulator(verbose=False):
    """Initialize EMANTIS emulator for f(R) gravity boost."""
    return NonLinearMGBoostEmulator(verbose=verbose)


def compute_fR_boost(emantis_emu, params, logfR0, k, aexp=1.0):
    """Compute f(R) gravity boost."""
    params_emantis = {
        "Omega_m": params["Omega_m"],
        "Omega_b": params["Omega_b"],
        "h": params["h"],
        "n_s": params["n_s"],
        "sigma8_lcdm": params["sigma8"],
        "logfR0": logfR0
    }
    return emantis_emu.predict_boost(params_emantis, aexp=aexp, k=k)


def create_power_spectrum_interpolator(k, pk):
    """Create log-log interpolator for P(k)."""
    return interp1d(
        np.log(k),
        np.log(pk + 1e-50),
        kind='cubic',
        bounds_error=False,
        fill_value='extrapolate'
    )


def check_k_bounds(k_vals, k_min, k_max, ell, spectrum_name, threshold=0.05):
    """Clip k values to safe bounds (simple version)."""
    return np.clip(k_vals, k_min, k_max)


# ============================================================
# Angular Power Spectra (Limber approximation)
# ============================================================

def compute_Cl_galaxy_auto(
    ell_array, pk_interp_log, z_grid, dNdz,
    chi_vals, H_vals, D_vals, D0, c_light,
    H0, Om_m, Om_lambda, b0=2.0,
    k_min=None, k_max=None
):
    """Galaxy auto spectrum C_ℓ^{gg}."""
    b_vals = bias_function(z_grid, H0, Om_m, Om_lambda, b0=b0)
    Wg = b_vals * dNdz

    Cl = np.zeros_like(ell_array, dtype=float)

    for i, ell in enumerate(ell_array):
        k_vals = (ell + 0.5) / chi_vals

        if k_min is not None and k_max is not None:
            k_safe = check_k_bounds(k_vals, k_min, k_max, ell, "C_ℓ^gg")
        else:
            k_safe = k_vals

        try:
            logP0 = pk_interp_log(np.log(k_safe))
            P0 = np.exp(logP0)
        except ValueError:
            P0 = np.ones_like(k_vals) * 1e-10

        P_kz = P0 * (D_vals / D0) ** 2

        integrand = (H_vals / c_light) * (Wg ** 2) * P_kz / (chi_vals ** 2)
        Cl[i] = simpson(integrand, z_grid)

    return Cl


def compute_Cl_galaxy_cmb_cross(
    ell_array, pk_interp_log, z_grid, dNdz,
    chi_vals, H_vals, D_vals, D0, Wkappa_vals, c_light,
    H0, Om_m, Om_lambda, b0=2.0,
    k_min=None, k_max=None
):
    """Galaxy–CMB lensing cross spectrum C_ℓ^{κg}."""
    b_vals = bias_function(z_grid, H0, Om_m, Om_lambda, b0=b0)
    Wg = b_vals * dNdz

    Cl = np.zeros_like(ell_array, dtype=float)

    for i, ell in enumerate(ell_array):
        k_vals = (ell + 0.5) / chi_vals

        if k_min is not None and k_max is not None:
            k_safe = check_k_bounds(k_vals, k_min, k_max, ell, "C_ℓ^κg")
        else:
            k_safe = k_vals

        try:
            logP0 = pk_interp_log(np.log(k_safe))
            P0 = np.exp(logP0)
        except ValueError:
            P0 = np.ones_like(k_vals) * 1e-10

        P_kz = P0 * (D_vals / D0) ** 2

        integrand = (H_vals / c_light) * (Wkappa_vals * Wg) * P_kz / (chi_vals ** 2)
        Cl[i] = simpson(integrand, z_grid)

    return Cl


# ============================================================
# Plotting
# ============================================================

def plot_power_spectra(ell, Cl_gg_gr, Cl_kg_gr, fr_results=None,
                        fR_values=None, figsize=(14, 10)):
    """Plot C_ℓ^gg, C_ℓ^{κg} and f(R)/GR ratios."""
    fig = plt.figure(figsize=figsize)

    # C_ℓ^{gg}
    plt.subplot(2, 2, 1)
    plt.loglog(ell, Cl_gg_gr, 'b-', linewidth=2.5,
                label=r'$C_\ell^{gg}\;[\mathrm{GR}]$')

    if fr_results is not None and fR_values is not None:
        for logfR0_val in fR_values:
            if logfR0_val in fr_results["gg"]:
                plt.loglog(
                    ell, fr_results["gg"][logfR0_val], '--', linewidth=2,
                    label=rf'$C_\ell^{{gg}}[f(R):\,\log_{10}|f_{{R0}}|=-{logfR0_val}]$'
                )

    plt.xlabel(r'$\ell$', fontsize=12)
    plt.ylabel(r'$C_\ell^{gg}$', fontsize=12)
    plt.title(r'Galaxy Auto Spectrum $C_\ell^{gg}$', fontsize=14)
    plt.legend(fontsize=9, loc='best')
    plt.grid(True, ls='--', alpha=0.7)

    # C_ℓ^{κg}
    plt.subplot(2, 2, 2)
    plt.loglog(ell, np.abs(Cl_kg_gr), 'b-', linewidth=2.5,
                label=r'$C_\ell^{\kappa g}\;[\mathrm{GR}]$')

    if fr_results is not None and fR_values is not None:
        for logfR0_val in fR_values:
            if logfR0_val in fr_results["kg"]:
                plt.loglog(
                    ell, np.abs(fr_results["kg"][logfR0_val]), '--', linewidth=2,
                    label=rf'$C_\ell^{{\kappa g}}[f(R):\,\log_{10}|f_{{R0}}|=-{logfR0_val}]$'
                )

    plt.xlabel(r'$\ell$', fontsize=12)
    plt.ylabel(r'$|C_\ell^{\kappa g}|$', fontsize=12)
    plt.title(r'CMB Lensing × Galaxy $C_\ell^{\kappa g}$', fontsize=14)
    plt.legend(fontsize=9, loc='best')
    plt.grid(True, ls='--', alpha=0.7)

    # Ratio C_ℓ^{gg}(fR)/C_ℓ^{gg}(GR)
    plt.subplot(2, 2, 3)
    if fr_results is not None and fR_values is not None:
        for logfR0_val in fR_values:
            if logfR0_val in fr_results["gg"]:
                ratio = fr_results["gg"][logfR0_val] / Cl_gg_gr
                plt.semilogx(ell, ratio, '--', linewidth=2,
                                label=rf'$\log_{10}|f_{{R0}}|=-{logfR0_val}$')
        plt.axhline(y=1.0, color='k', linestyle='-', linewidth=1.5, alpha=0.5)

    plt.xlabel(r'$\ell$', fontsize=12)
    plt.ylabel(r'$C_\ell^{gg}(f(R))/C_\ell^{gg}(\mathrm{GR})$', fontsize=12)
    plt.title(r'Galaxy Auto Ratio', fontsize=14)
    plt.legend(fontsize=9, loc='best')
    plt.grid(True, ls='--', alpha=0.7)
    plt.ylim([0.95, 1.45])

    # Ratio C_ℓ^{κg}(fR)/C_ℓ^{κg}(GR)
    plt.subplot(2, 2, 4)
    if fr_results is not None and fR_values is not None:
        for logfR0_val in fR_values:
            if logfR0_val in fr_results["kg"]:
                ratio = fr_results["kg"][logfR0_val] / Cl_kg_gr
                plt.semilogx(ell, ratio, '--', linewidth=2,
                                label=rf'$\log_{10}|f_{{R0}}|=-{logfR0_val}$')
        plt.axhline(y=1.0, color='k', linestyle='-', linewidth=1.5, alpha=0.5)

    plt.xlabel(r'$\ell$', fontsize=12)
    plt.ylabel(r'$C_\ell^{\kappa g}(f(R))/C_\ell^{\kappa g}(\mathrm{GR})$', fontsize=12)
    plt.title(r'Galaxy–CMB Cross Ratio', fontsize=14)
    plt.legend(fontsize=9, loc='best')
    plt.grid(True, ls='--', alpha=0.7)
    plt.ylim([0.95, 1.45])

    plt.tight_layout()
    return fig


# ============================================================
# Initialize Cosmology
# ============================================================

H0 = Config.COSMO_PARAMS["h"] * 100.0  # km/s/Mpc
Om_m = Config.COSMO_PARAMS["Omega_m"]
Om_lambda = 1.0 - Om_m  # flat ΛCDM assumption for this run


# ============================================================
# Matter Power Spectra
# ============================================================

k, pk_nl_gr, Q_boost = initialize_bacco_emulator(
    Config.COSMO_PARAMS,
    k_min=Config.BACCO_K_MIN,
    k_max=Config.BACCO_K_MAX,
    n_k=Config.BACCO_N_K,
    verbose=Config.EMANTIS_VERBOSE
)
pk_interp_gr = create_power_spectrum_interpolator(k, pk_nl_gr)

emantis_emu = initialize_emantis_emulator(verbose=Config.EMANTIS_VERBOSE)


# ============================================================
# Pre-compute Redshift-Dependent Quantities
# ============================================================

z_grid = np.linspace(Config.Z_MIN, Config.Z_MAX, Config.N_Z)

# --- Normalized galaxy redshift distribution dN/dz ---
# Replacement for removed functions: Raw n(z) ∝ exp(-z / z0)
_raw_nz = np.exp(-z_grid / Config.Z0_GALAXY)
_norm_constant = simpson(_raw_nz, z_grid)
dNdz = _raw_nz / _norm_constant

chi_vals = comoving_distance_proper(z_grid, H0, Om_m, Om_lambda)
H_vals = hubble_function(z_grid, H0, Om_m, Om_lambda)

D_vals = growth_factor_D(z_grid, H0, Om_m, Om_lambda)
D0 = growth_factor_D(0.0, H0, Om_m, Om_lambda)

# CMB lensing kernel at each z
z_star = 1100.0
Wkappa_vals = cmb_lensing_kernel(z_grid, z_star, H0, Om_m, Om_lambda)

# ============================================================
# Angular Power Spectra for GR
# ============================================================

ell = np.logspace(np.log10(Config.ELL_MIN),
                  np.log10(Config.ELL_MAX),
                  Config.N_ELL)

k_min_safe = k.min() * Config.K_MIN_SAFETY
k_max_safe = k.max() * Config.K_MAX_SAFETY

Cl_gg_gr = compute_Cl_galaxy_auto(
    ell, pk_interp_gr, z_grid, dNdz,
    chi_vals, H_vals, D_vals, D0, Config.c,
    H0, Om_m, Om_lambda, b0=Config.BIAS_B0,
    k_min=k_min_safe, k_max=k_max_safe
)

Cl_kg_gr = compute_Cl_galaxy_cmb_cross(
    ell, pk_interp_gr, z_grid, dNdz,
    chi_vals, H_vals, D_vals, D0, Wkappa_vals, Config.c,
    H0, Om_m, Om_lambda, b0=Config.BIAS_B0,
    k_min=k_min_safe, k_max=k_max_safe
)


# ============================================================
# Angular Power Spectra for f(R)
# ============================================================

fr_results = {"gg": {}, "kg": {}}

for logfR0_val in Config.FR_VALUES:
    pk_boost = compute_fR_boost(
        emantis_emu, Config.COSMO_PARAMS,
        logfR0_val, k,
        aexp=Config.COSMO_PARAMS["aexp"]
    )
    pk_fR = pk_nl_gr * pk_boost
    pk_interp_fr = create_power_spectrum_interpolator(k, pk_fR)

    Cl_gg_fr = compute_Cl_galaxy_auto(
        ell, pk_interp_fr, z_grid, dNdz,
        chi_vals, H_vals, D_vals, D0, Config.c,
        H0, Om_m, Om_lambda, b0=Config.BIAS_B0,
        k_min=k_min_safe, k_max=k_max_safe
    )

    Cl_kg_fr = compute_Cl_galaxy_cmb_cross(
        ell, pk_interp_fr, z_grid, dNdz,
        chi_vals, H_vals, D_vals, D0, Wkappa_vals, Config.c,
        H0, Om_m, Om_lambda, b0=Config.BIAS_B0,
        k_min=k_min_safe, k_max=k_max_safe
    )

    fr_results["gg"][logfR0_val] = Cl_gg_fr
    fr_results["kg"][logfR0_val] = Cl_kg_fr


# ============================================================
# Plot Results
# ============================================================

fig = plot_power_spectra(
    ell, Cl_gg_gr, Cl_kg_gr,
    fr_results=fr_results,
    fR_values=Config.FR_VALUES,
    figsize=Config.FIGURE_SIZE
)
plt.show()

print("Analysis complete!")